In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             average_precision_score, confusion_matrix)

DATA_PATH = 'C:\\Users\\amanm\\OneDrive\\Desktop\\dont open\\Career\\Breakthrough Team\\Personal\\data\\fraud_oracle.csv'   # change to your local path
TARGET = 'FraudFound_P'

In [2]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df[TARGET].value_counts(normalize=True))

(15420, 33)
FraudFound_P
0    0.940143
1    0.059857
Name: proportion, dtype: float64


Cleaning

In [3]:
# one row has '0' as a placeholder in DayOfWeekClaimed and MonthClaimed
bad = (df['DayOfWeekClaimed'] == '0') | (df['MonthClaimed'] == '0')
print('rows dropped:', bad.sum())
df = df[~bad].reset_index(drop=True)

# PolicyNumber is an ID that lines up with year, so it could leak timing information
df = df.drop(columns=['PolicyNumber'])

# Age == 0 is a placeholder, not a real age: flag it and treat it as missing.
# No rows are removed. The flag is safe to create before the split (it only reads the row itself).
# The actual imputation happens AFTER the split, using the training-set median.
df['age_was_missing'] = (df['Age'] == 0).astype(int)
df['Age'] = df['Age'].replace(0, np.nan)
print('Age missing:', df['age_was_missing'].sum())
print(df.shape)

rows dropped: 1
Age missing: 319
(15419, 33)


In [4]:
LEAKAGE_DROP = ['PoliceReportFiled', 'WitnessPresent',
                'NumberOfSuppliments', 'AddressChange_Claim']
# add more here based on audit

df_model = df.drop(columns=LEAKAGE_DROP)
print(df_model.shape)

(15419, 29)


In [5]:
summary = df_model.groupby('Year')[TARGET].agg(rows='count', fraud='sum', fraud_rate='mean')
print(summary)

      rows  fraud  fraud_rate
Year                         
1994  6141    409    0.066602
1995  5195    301    0.057940
1996  4083    213    0.052168


In [6]:
train = df_model[df_model['Year'] == 1994].copy()
val   = df_model[df_model['Year'] == 1995].copy()
test  = df_model[df_model['Year'] == 1996].copy()

# impute missing Age using the TRAIN median only, then apply the same value everywhere
age_median = train['Age'].median()   # NaNs are skipped
print('train median Age:', age_median)
for part in (train, val, test):
    part['Age'] = part['Age'].fillna(age_median)

feature_cols = [c for c in df_model.columns if c not in (TARGET, 'Year')]

X_train, y_train = train[feature_cols], train[TARGET]
X_val,   y_val   = val[feature_cols],   val[TARGET]
X_test,  y_test  = test[feature_cols],  test[TARGET]

for name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f'{name}: {len(y)} rows, {y.sum()} fraud ({y.mean():.3%})')

train median Age: 38.0
train: 6141 rows, 409 fraud (6.660%)
val: 5195 rows, 301 fraud (5.794%)
test: 4083 rows, 213 fraud (5.217%)


In [7]:
def top_k_capture(y_true, scores, k):
    # share of all fraud cases found if only the top k fraction of claims (by score) is reviewed
    y_true = np.asarray(y_true)
    n = int(np.ceil(len(y_true) * k))
    top_idx = np.argsort(-np.asarray(scores))[:n]
    return y_true[top_idx].sum() / y_true.sum()


def evaluate(y_true, scores, threshold=0.5, name='model', verbose=True):
    preds = (np.asarray(scores) >= threshold).astype(int)
    out = {
        'model': name,
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall': recall_score(y_true, preds),
        'f1': f1_score(y_true, preds),
        'pr_auc': average_precision_score(y_true, scores),
        'capture_top10': top_k_capture(y_true, scores, 0.10),
        'capture_top20': top_k_capture(y_true, scores, 0.20),
    }
    if verbose:
        print(f"--- {name} (threshold={threshold}) ---")
        for k, v in out.items():
            if k != 'model':
                print(f'{k:>14}: {v:.3f}')
        print(confusion_matrix(y_true, preds))
    return out

In [8]:
rng = np.random.default_rng(42)
naive_scores = rng.random(len(y_val))
results = [evaluate(y_val, naive_scores, threshold=0.5, name='naive (random)')]
print('val fraud rate (PR-AUC floor):', round(y_val.mean(), 4))

--- naive (random) (threshold=0.5) ---
     precision: 0.060
        recall: 0.512
            f1: 0.108
        pr_auc: 0.065
 capture_top10: 0.096
 capture_top20: 0.213
[[2497 2397]
 [ 147  154]]
val fraud rate (PR-AUC floor): 0.0579


## Creating the actual model

In [10]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Identify categorical and numerical features from X_train
categorical_cols = X_train.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

# Encode categorical features and scale numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical',
         OneHotEncoder(handle_unknown='ignore'),
         categorical_cols),

        ('numerical',
         StandardScaler(),
         numerical_cols)
    ]
)

# Combine preprocessing and logistic regression
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train the complete pipeline
model.fit(X_train, y_train)


C:\Users\amanm\AppData\Local\Temp\ipykernel_43728\3926449206.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


Categorical columns: ['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'AgentType', 'NumberOfCars', 'BasePolicy']
Numerical columns: ['WeekOfMonth', 'WeekOfMonthClaimed', 'Age', 'RepNumber', 'Deductible', 'DriverRating', 'age_was_missing']


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](27,)","['Month','WeekOfMonth','DayOfWeek',...,'NumberOfCars','BasePolicy', 'age_was_missing']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,27
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying

In [11]:
# evaluate using our function
val_scores = model.predict_proba(X_val)[:, 1]

results.append(
    evaluate(y_val, val_scores, name='logistic regression')
)

--- logistic regression (threshold=0.5) ---
     precision: 0.056
        recall: 0.003
            f1: 0.006
        pr_auc: 0.101
 capture_top10: 0.169
 capture_top20: 0.349
[[4877   17]
 [ 300    1]]
